# Машинное обучение, ФКН ВШЭ

# Практическое задание 11. Несбалансированные задачи и поиск аномалий

## Общая информация
Дата выдачи: 21.11.2026

Мягкий дедлайн: 06.12.2026 23:59 MSK

Жесткий дедлайн: 13.12.2026 23:59 MSK

## Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за работу — 10 баллов. Бонусные баллы могут компенсировать потерянные в основной части.

Сдавать задание после указанного срока сдачи нельзя. При выставлении неполного балла за задание в связи с наличием ошибок на усмотрение проверяющего предусмотрена возможность исправить работу на указанных в ответном письме условиях.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке.

## Формат сдачи
Посылка должна содержать:
* Ноутбук homework-practice-11-Username.ipynb

Username — ваша фамилия и имя на латинице именно в таком порядке.

## О задании

В этом задании мы разберем задачи, где один из классов занимает очень малую долю выборки. Для простоты ограничимся бинарной задачей, хотя большинство идей переносится и на многоклассовый случай.

Задание состоит из двух частей:

1. **Несбалансированная классификация.** Разберем, какие есть основные приемы для работы с несбалансированными данными и работают ли они вообще.
2. **Поиск аномалий.** Реализуем с нуля Isolation Forest и его обобщение Extended Isolation Forest и сравним их с другими методами поиска аномалий.

Везде в задании, где требуется сделать вывод, пишите его текстом.

In [ ]:
!pip install -q catboost imbalanced-learn "scikit-learn>=1.5"

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

from tqdm.auto import tqdm

import sklearn
print(sklearn.__version__)

**Задание 0 (1 балл)**. EDA и разбиение данных

В качестве данных возьмем датасет транзакций по банковским картам, в котором нужно находить мошеннические операции ([описание на kaggle](https://www.kaggle.com/mlg-ulb/creditcardfraud)). Данные несбалансированны сами по себе - среди всех транзакций клиентов мошеннических очень мало.

Чтобы не возиться с авторизацией на kaggle, загрузим копию датасета напрямую (около 150 МБ).

In [ ]:
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
df = pd.read_csv(DATA_URL)
df

Данные анонимизированы. У нас 30 признаков: 28 из них — результат PCA-преобразования исходных данных, еще 2 — время в секундах с момента первой транзакции в датасете (`Time`) и сумма транзакции (`Amount`). Какова доля положительных объектов в выборке?

In [ ]:
# your code here

Секунды сами по себе мало что говорят о зависимостях в данных. Создайте по ним признаки «час» (от 0 до 23) и «день» (от 0 до ...), принимая первый объект выборки за начальную точку. Сколько дней покрывают данные?

In [ ]:
# your code here

Постройте следующие графики:

1. Число транзакций по каждому часу (line-plot).
2. Доля мошеннических транзакций по каждому часу (line-plot).
3. То же самое для дней (здесь можно использовать bar-plot, так как дней немного).

Какие выводы можно сделать из графиков? Как, на ваш взгляд, полученные часы связаны с реальным временем суток?

In [ ]:
# your code here

С анонимизированными признаками вряд ли получится придумать что-то содержательное. Выберите несколько наиболее информативных признаков (например, по корреляции с таргетом) и сравните их распределения для разных классов.

In [ ]:
# your code here

Теперь давайте разделим данные. Отделите хронологически последние 20% транзакций и поделите их пополам (также хронологически, т.е. без перемешивания) на валидационные и тестовые. Это разбиение не совсем корректно (как можно было заметить, мошеннические транзакции имеют разное распределение во времени - по-хорошему, нам стоило бы выделить целые сутки записей как под валидацию, так и под тест), тем не менее, мы не сможем получить больше данных для адекватного контроля, поэтому обойдемся этим.

In [ ]:
# your code here

# Часть 1. Несбалансированная классификация

**Задание 1 (1 балл)**. Метрики

Классические метрики качества классификации часто «ломаются» при сильном перекосе классов. Чему равна accuracy наивного предсказания (мажорный класс для каждого объекта)?

Из курса МО-1 вы знаете, что в таких задачах можно использовать `PR-AUC`. Рассмотрим еще две метрики, которые считаются по бинарным предсказаниям.

**Каппа Коэна** — нормализованная `accuracy`:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Метрика измеряет согласованность двух независимых предсказателей и не различает «верные» и «предсказанные» метки. Здесь $p_o$ — доля совпадающих ответов (accuracy), а $p_e$ — доля согласованных предсказаний, которая могла бы получиться при случайных ответах предсказателей. Метрика принимает значения от -1 до 1: 1 означает полную согласованность, 0 — согласованность на уровне случайной, -1 — полную несогласованность.

**Коэффициент корреляции Мэтьюза**:

$$\text{MCC} = \frac{TP\times TN - FP \times FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

Значения также лежат от -1 до 1 и интерпретируются аналогично. Если знаменатель равен нулю, будем считать $\text{MCC} = 0$.

Реализуйте обе метрики и убедитесь, что тесты ниже проходят.

In [ ]:
def cohen_kappa(y_true, y_pred):
    # your code here
    pass


def mcc(y_true, y_pred):
    # your code here
    pass

In [ ]:
# Тесты. Не изменяйте эту ячейку
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef

_rng = np.random.default_rng(0)
for _ in range(20):
    _a = (_rng.random(1000) < _rng.uniform(0.01, 0.5)).astype(int)
    _b = (_rng.random(1000) < _rng.uniform(0.01, 0.5)).astype(int)
    assert np.isclose(cohen_kappa(_a, _b), cohen_kappa_score(_a, _b))
    assert np.isclose(cohen_kappa(_a, _b), cohen_kappa(_b, _a))
    assert np.isclose(mcc(_a, _b), matthews_corrcoef(_a, _b))
assert mcc(_a, np.zeros_like(_a)) == 0
print('OK')

Вычислите accuracy, $\kappa$ и $\text{MCC}$ для наивного предсказания на тестовой выборке.

In [ ]:
# your code here

Теперь сравним `ROC-AUC` и `PR-AUC`. Посчитайте обе метрики на тестовой выборке для случайных скоров (например, равномерных на $[0, 1]$). Чему равны их значения и почему?

Представьте модель, у которой `ROC-AUC = 0.95`. Объясните через определения FPR и precision, почему при доле положительных около 0.1% такая модель может давать очень много ложных тревог на каждое верное срабатывание. Какая из двух метрик лучше отражает это для нашей задачи?

In [ ]:
# your code here

**Задание 2 (1.5 балла)**. Контрольная модель и выбор порога

Для начала построим контрольную модель. Обучите `CatBoostClassifier` на всех признаках **без каких-либо весов и ресэмплинга**. Для контроля переобучения используйте валидационную выборку.

Вычислите на тестовой выборке `PR-AUC`, `ROC-AUC`, а также $\kappa$ и $\text{MCC}$ при стандартном пороге 0.5.

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# your code here

Метрики $\kappa$ и $\text{MCC}$ зависят от порога бинаризации, и порог 0.5 ничем не обоснован. Подберите порог, максимизирующий $\text{MCC}$ на **валидационной** выборке, с помощью [`TunedThresholdClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TunedThresholdClassifierCV.html). Модель уже обучена, а валидация отделена хронологически, поэтому используйте `cv="prefit"` и `refit=False`, а `fit` вызывайте на валидационных данных.

Какой порог получился? Как изменились $\kappa$ и $\text{MCC}$ на тесте? Изменились ли `PR-AUC` и `ROC-AUC` и почему?

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import make_scorer

In [ ]:
# your code here

**Задание 3 (2 балла)**. Помогает ли балансировка?

Существует много способов «исправить» дисбаланс классов: веса в функции потерь, оверсэмплинг, андерсэмплинг. В этом задании проверим, дают ли они что-то по сравнению с контрольной моделью из задания 2.

Правила сравнения для всех методов ниже:
- ресэмплинг применяется **только к обучающей выборке**, валидационная и тестовая не изменяются (мы не хотим измерять качество на объектах, которых в реальности не существует);
- для каждого метода порог подбирается на валидации так же, как в задании 2;
- на тесте считаются `PR-AUC`, `ROC-AUC`, $\text{MCC}$ при подобранном пороге и стоимость ошибок при подобранном пороге.

**3.1 (0.5 балла)**. Веса классов.

В `catboost` вес минорного класса можно задать через `scale_pos_weight` или `class_weights`. Чаще всего в качестве веса берут отношение числа объектов мажорного класса к числу объектов минорного. Обучите модель с таким весом и сравните с контрольной моделью.

Подберите вес на валидационной выборке по `PR-AUC`, включая значения меньше 1 (то есть фактически увеличение веса мажорного класса) и экстремально большие. Какой вес оказался оптимальным?

In [ ]:
# your code here

**3.2 (0.5 балла)**.Oversampling

Принцип работы: для объекта минорного класса выбирается случайный сосед среди $k$ ближайших объектов того же класса, и в выборку добавляется случайная точка на отрезке между ними.

Используйте библиотеку [imbalanced-learn](https://imbalanced-learn.org/stable/). Соберите `SMOTE` и `CatBoostClassifier` в `imblearn.pipeline.Pipeline` и сравните полное выравнивание классов с частичным: 1:1, 1:2 и 1:10 (параметр `sampling_strategy`). Веса классов здесь не используйте.


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [ ]:
# your code here

**3.3 (0.5 балла)**. Undersampling.

Возможно, для восстановления закономерностей нам не нужно столько объектов мажорного класса. Самое простое — случайно удалять объекты мажорного класса до приемлемого баланса. Протестируйте `RandomUnderSampler` и постройте графики метрик в зависимости от итогового баланса классов и от доли оставшихся объектов.

Затем протестируйте алгоритм [Near-Miss](https://www.site.uottawa.ca/~nat/Workshop2003/jzhang.pdf), который удаляет объекты мажорного класса с учетом их близости к минорному. Постройте аналогичные графики и добавьте зависимость качества от числа соседей. Сравните версии алгоритма.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler, NearMiss

In [ ]:
# your code here

**3.4 (0.5 балла)**. Выводы

1. Постройте калибровочные кривые (`sklearn.calibration.CalibrationDisplay`) на тестовой выборке для контрольной модели и для лучших моделей из пунктов 3.1–3.3. Посчитайте для них Brier score и log-loss. Что происходит с вероятностями после балансировки и почему?
2. Оцените разброс PR-AUC с помощью бутстрапа на  тестовой выборке. Напишите вывод: помогает ли балансировка в этой задаче для этой модели? Какой метод вы бы выбрали на практике и почему?

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss, log_loss

In [ ]:
# your code here

# Часть 2. Поиск аномалий

Сменим постановку задачи и перейдем к поиску *аномалий*. Мошеннические транзакции вполне естественно считать аномальными: их мало, и они должны отличаться от обычных.

Методы поиска аномалий обычно обучаются без учителя. Это удобно: не нужно размечать данные (хотя для контроля качества небольшую разметку сделать все равно придется). Платить за это приходится качеством: как правило, такие методы заметно уступают обучению с учителем, если разметка доступна.

**Задание 4 (2.5 балла)**. Isolation Forest с нуля

Напомним идею Isolation Forest. Строим ансамбль деревьев, в каждой вершине которых признак и порог выбираются случайно (порог — равномерно между минимумом и максимумом признака на объектах вершины). Аномальные объекты отделяются от остальных быстрее, поэтому глубина листа, в который попадает объект, в среднем меньше. Оценка аномальности:

$$s(x, n) = 2^{-\frac{\mathbb{E}\, h(x)}{c(n)}},$$

где $h(x)$ — длина пути до листа (глубина листа плюс поправка $c(\text{размер листа})$ для листов, которые не удалось разделить до конца), $n$ — размер подвыборки для построения дерева, а $c(n)$ — средняя длина пути неуспешного поиска в бинарном дереве поиска:

$$c(n) = 2 H(n - 1) - \frac{2(n - 1)}{n}, \quad H(i) \approx \ln i + 0.5772156649,$$

$c(2) = 1$, $c(n) = 0$ при $n \le 1$.

[Extended Isolation Forest](https://arxiv.org/abs/1811.02141) вместо порога по одному признаку использует случайную гиперплоскость: выбирается вектор нормали $\vec{n}$ со случайными компонентами из $\mathcal{N}(0, 1)$ и точка $\vec{p}$, равномерно распределенная в ограничивающем прямоугольнике объектов вершины. Объект $\vec{x}$ отправляется влево, если $(\vec{x} - \vec{p}) \cdot \vec{n} \le 0$.

`extension_level` задает, сколько координат нормали отличны от нуля: у вектора $\vec{n}$ случайно выбранные $d - \text{extension\_level} - 1$ координат зануляются, где $d$ — число признаков. При `extension_level = 0` остается одна ненулевая координата, и мы получаем обычный Isolation Forest. При `extension_level = d - 1` разрез — гиперплоскость общего положения.

Реализуйте метод в рамках шаблона ниже. Будем считать, что категориальных признаков нет.

**Требования и советы**:
- Не обходите объекты циклом по одному: при построении и применении дерева передавайте в рекурсию массивы индексов объектов, попавших в вершину. Ориентир: на обучающей выборке этого датасета 100 деревьев строятся и применяются к 50 тысячам объектов за несколько секунд.
- Не игнорируйте документирующие строки.
- Внимательно следите за знаками: в `sklearn` метод `score_samples` возвращает величину, которая *больше* для нормальных объектов.

In [ ]:
EULER_GAMMA = 0.5772156649


def c_factor(n):
    """
    Average path length of an unsuccessful search in a binary search tree.

    Params:
        n: int or np.array of ints - number of points
    Returns:
        float or np.array of floats of the same shape as n
    """
    # your code here


class Node:
    """
    Node of an isolation tree.

    Internal nodes store the splitting hyperplane: a point x goes to the left child
    if (x - intercept) @ normal <= 0, and to the right child otherwise.
    Leaves store the number of training points that reached them and their depth.
    """
    def __init__(self, normal=None, intercept=None, left=None, right=None, size=0, depth=0):
        self.normal = normal
        self.intercept = intercept
        self.left = left
        self.right = right
        self.size = size
        self.depth = depth

    @property
    def is_leaf(self):
        return self.left is None


class IsolationTree:
    def __init__(self, height_limit, extension_level, rng):
        """
        Params:
            height_limit: int - maximum depth of the tree
            extension_level: int - number of non-zero coordinates of the normal vector minus one
            rng: np.random.Generator - source of randomness, use it for all random choices
        """
        self.height_limit = height_limit
        self.extension_level = extension_level
        self.rng = rng

    def _random_split(self, X):
        """
        Samples a random hyperplane for the points X available to the node.

        Returns:
            normal: np.array of shape (n_features,) with exactly extension_level + 1 non-zero coordinates
            intercept: np.array of shape (n_features,) - uniform point in the bounding box of X
        """
        # your code here

    def _build(self, X, depth):
        """
        Recursively builds the subtree for the points X at the given depth.
        A node becomes a leaf if depth reaches height_limit or it contains at most one point.

        Returns:
            Node
        """
        # your code here

    def fit(self, X):
        self.root = self._build(X, 0)
        return self

    def path_length(self, X):
        """
        Computes h(x) for every row of X: depth of the reached leaf plus c_factor(leaf size).

        Returns:
            np.array of shape (n_samples,)
        """
        # your code here


class ExtendedIsolationForest:
    def __init__(self, n_estimators=100, max_samples=256, extension_level=0, random_state=None):
        """
        Params:
            n_estimators: int - number of trees
            max_samples: int - size of the subsample (without replacement) used to build each tree
            extension_level: int from 0 to n_features - 1; 0 gives the classic Isolation Forest
            random_state: int or None - seed for np.random.default_rng
        """
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.extension_level = extension_level
        self.random_state = random_state

    def fit(self, X):
        """
        Builds the forest. The height limit of each tree is ceil(log2(max_samples)).
        Raises ValueError if extension_level is outside [0, n_features - 1].
        """
        # your code here
        return self

    def anomaly_score(self, X):
        """
        Returns:
            np.array of shape (n_samples,) with s(x, max_samples) in (0, 1); larger means more anomalous
        """
        # your code here

    def score_samples(self, X):
        """Same convention as in sklearn: larger means more normal."""
        return -self.anomaly_score(X)

In [ ]:
# Тесты. Не изменяйте эту ячейку
from scipy.stats import spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.ensemble._iforest import _average_path_length

_ns = np.array([0, 1, 2, 3, 10, 256, 1000])
assert np.allclose(c_factor(_ns), _average_path_length(_ns))

_rng = np.random.default_rng(0)
_X = np.vstack([_rng.normal(size=(2000, 5)), _rng.normal(4, 1, size=(40, 5))])

for _seed in range(3):
    _ours = ExtendedIsolationForest(500, 256, 0, random_state=_seed).fit(_X).anomaly_score(_X)
    _sk = -IsolationForest(n_estimators=500, max_samples=256, random_state=_seed).fit(_X).score_samples(_X)
    assert _ours.shape == (len(_X),) and np.all((_ours > 0) & (_ours < 1))
    assert np.abs(_ours - _sk).mean() < 0.015, 'extension_level=0 должен совпадать с обычным IF'
    assert spearmanr(_ours, _sk)[0] > 0.98, 'extension_level=0 должен совпадать с обычным IF'

_X_out = np.vstack([_X, np.full((1, 5), 10.0)])
for _ext in [0, 2, 4]:
    _s = ExtendedIsolationForest(200, 256, _ext, random_state=0).fit(_X_out).anomaly_score(_X_out)
    assert np.argmax(_s) == len(_X_out) - 1, 'явный выброс должен получать максимальную оценку'

for _bad in [-1, 5]:
    try:
        ExtendedIsolationForest(extension_level=_bad).fit(_X)
        raise AssertionError('ожидалась ошибка для extension_level=%d' % _bad)
    except ValueError:
        pass
print('OK')

Визуализируем разницу между IF и EIF. Сгенерируйте двумерную выборку из двух гауссовских облаков (например, с центрами в $(0, 0)$ и $(5, 5)$) и постройте карты оценки аномальности на сетке для `extension_level = 0` и `extension_level = 1`. Где у обычного IF возникают «призрачные» области низкой аномальности? Объясните, откуда они берутся.

In [ ]:
# your code here

**Бонус (0.5 балла)**. Ускорьте построение и применение леса: распараллельте его по деревьям (`joblib`) и/или полностью векторизуйте обход дерева (например, храня дерево в массивах, а не в объектах `Node`). Покажите замеры времени до и после.

In [ ]:
# your code here

**Задание 5 (2 балла)**. Сравнение моделей

**5.1 (0.75 балла)**. Протестируйте вашу реализацию на наших данных. Обучайте модели на обучающей выборке, а гиперпараметры (`n_estimators`, `max_samples`, `extension_level`, выбор признаков, предобработку) подбирайте по `PR-AUC` на валидационной. Сравните с `IsolationForest` из `sklearn` при одинаковых параметрах. Удалось ли с помощью `extension_level > 0` улучшить качество?

Проверьте, как влияет стандартизация признаков на IF и на EIF с максимальным `extension_level`. Объясните результат.

In [ ]:
# your code here

**5.2 (0.5 балла)**. `sklearn` предлагает и другие методы поиска аномалий. Протестируйте [Local Outlier Factor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html) (с `novelty=True`) и [One-Class SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.OneClassSVM.html). Оба метода плохо масштабируются, поэтому обучайте их на случайной подвыборке обучающей выборки разумного размера. Постройте PR-кривые всех детекторов (IF, лучший EIF, LOF, OCSVM) и контрольной модели из задания 2 на одном графике.

In [ ]:
# your code here

**5.3 (0.25 балла)**. Скорее всего, ваш лучший детектор аномалий выдает высокие оценки аномальности и некоторым обычным транзакциям. Выберите несколько таких объектов и объясните (с кодом и графиками), почему так вышло.

In [ ]:
# your code here

**5.4 (0.5 балла)**. Используем детекторы как источник признаков. Добавьте оценки аномальности всех методов из этой части как новые признаки (детекторы обучаются только на обучающей выборке) и обучите на расширенных данных контрольную модель из задания 2. Смогли ли методы без учителя повысить качество?

In [ ]:
# your code here

# Бонусы

**Бонус (1 балл)**. Deep Isolation Forest. В [статье](https://arxiv.org/abs/2206.06602) предложено пропускать данные через случайно инициализированные нейросети и строить обычные осепараллельные деревья в полученных представлениях. Так разбиение исходного пространства становится нелинейным. Реализуйте упрощенную версию поверх своего Isolation Forest: для каждой группы деревьев сгенерируйте случайный MLP (веса не обучаются), отобразите им данные и постройте лес на представлениях. Сравните с IF и EIF на двумерном примере из задания 4 и на наших данных.

In [ ]:
# your code here

**Бонус (0.5 балла)**. Focal loss уменьшает вклад объектов, которые модель уже уверенно классифицирует правильно, и тем самым фокусирует обучение на трудных объектах. Выпишите формулу и объясните роль параметров $\alpha$ и $\gamma$. Обучите `catboost` с `loss_function='Focal:focal_alpha=...;focal_gamma=...'`, подберите параметры и сравните с контрольной моделью по правилам задания 3, включая калибровку.

In [ ]:
# your code here